### **Setup: Imports und Initialisierung**

In [15]:
import sys
import os

sys.path.append(os.path.abspath('..'))

from quinex_application.quinex_discussion import QuinexDiscussion

In [16]:
discussion = QuinexDiscussion()

### **Load Results**

In [17]:
df_quinex = discussion.load_df("../data/quinex/df_quinex.parquet")
df_quinex_mapped = discussion.load_df("../data/quinex/df_quinex_prop_ent_mapped.parquet")
print(f"Proportion of mapped statements: {len(df_quinex)}/{len(df_quinex_mapped)}")

📂 Loaded DataFrame from '../data/quinex/df_quinex.parquet' with 1282088 rows
📂 Loaded DataFrame from '../data/quinex/df_quinex_prop_ent_mapped.parquet' with 5535 rows
Proportion of mapped statements: 1282088/5535


### **Results Metadata**

### Selected Technologies

In [18]:
result = discussion.get_metadata(
    df_quinex,
    entity_keywords=['battery', 'batteries', 'hydrogen', 'pv', 'wind turbine', 'natural gas', 'electrolyzer', 'electrolyser'],
    property_keywords=['cost', 'price', 'LCOE', 'LCOH', 'COE', 'LCOS', 'price'],
)

# Ausgabe
print("Total Statements:", result['total_statements'])
top_entities_names = result['top_entities'].index.tolist()
top_properties_names = result['top_properties'].index.tolist()

technologies_df = result["filtered_df"]

Total Statements: 13030


In [19]:
canonical_entities = [item for key, values in discussion.entity_keywords.items() if key != "bess" for item in values]

mention_to_entity = discussion.map_mentions(top_entities_names, canonical_entities, threshold=0.8)
technologies_df = result["filtered_df"]
technologies_df["entity_mapping"] = technologies_df["entity"].map(mention_to_entity)
technologies_df = technologies_df[technologies_df["entity_mapping"].notna()]

In [20]:
canonical_properties = discussion.property_keywords["cost"]

mention_to_property = discussion.map_mentions(top_properties_names, canonical_properties, threshold=0.75)
technologies_df["property_mapping"] = technologies_df["property"].map(mention_to_property)
technologies_df = technologies_df[technologies_df["property_mapping"].notna()]

C:\Users\m.gorres\AppData\Local\Temp\ipykernel_14644\982125583.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  technologies_df["property_mapping"] = technologies_df["property"].map(mention_to_property)


In [ ]:
technologies_df["property"].value_counts()

In [21]:
property_keywords = {
    "Levelized Cost": ["LCOH", "COE", "LCOS", "LCOE", "Levelized Cost"]
}

def map_property(prop_value):
    for category, aliases in property_keywords.items():
        if prop_value in aliases:
            return category
    return prop_value

technologies_df["property_mapping"] = technologies_df["property_mapping"].apply(map_property)
technologies_df["property_mapping"].value_counts()

property_mapping
Cost                975
Levelized Cost      284
Capital Cost        173
Production Cost     127
Electricity Cost    103
Operating Cost       95
Total Cost           34
Name: count, dtype: int64

In [ ]:
def map_entity(entity_value):
    for category, keywords in discussion.entity_keywords.items():
        if entity_value in keywords:
            return category
    return entity_value  # Falls kein Match gefunden wird

technologies_df["entity_mapping"] = technologies_df["entity_mapping"].apply(map_entity)
technologies_df["entity_mapping"].value_counts()

property_mapping
Cost                975
Levelized Cost      284
Capital Cost        173
Production Cost     127
Electricity Cost    103
Operating Cost       95
Total Cost           34
Name: count, dtype: int64

In [25]:
discussion.save_df(technologies_df, "../data/visualisation_files/general/df_technologies_cost.parquet")

💾 Saved DataFrame to '../data/visualisation_files/general/df_technologies_cost.parquet' with 1791 rows


### Battery

In [ ]:
result = discussion.get_metadata(
    df_quinex,
    entity_keywords=['battery', 'batteries'],
    #property_keywords=['cost'],
    #top_n=80,
    exact_match=False
)

# Ausgabe
print("Total Statements:", result['total_statements'])

top_entities_names = result['top_entities'].index.tolist()
print("Number of Entities:", len(top_entities_names))

top_properties_names = result['top_properties'].index.tolist()
print("Number of Properties:", len(top_properties_names))

battery_df = result["filtered_df"]

In [ ]:
canonical_entities = discussion.entity_keywords["battery"]

mention_to_entity = discussion.map_mentions(top_entities_names, canonical_entities, threshold=0.8)
battery_df["entity_mapping"] = battery_df["entity"].map(mention_to_entity)
battery_df = battery_df[battery_df["entity_mapping"].notna()]

# Ausgabe
print("Total Statements:", len(battery_df))
print("Number of Entities:", len(battery_df["entity"].value_counts()))

In [ ]:
canonical_entities = discussion.property_keywords["efficiency"]

battery_df_efficiency = battery_df.copy()
mention_to_property = discussion.map_mentions(top_properties_names, canonical_entities, threshold=0.6)
battery_df_efficiency["property_mapping"] = battery_df_efficiency["property"].map(mention_to_property)
battery_df_efficiency = battery_df_efficiency[battery_df_efficiency["property_mapping"].notna()]

# Ausgabe
print("Total Statements:", len(battery_df_efficiency))
print("Number of Properties:", len(battery_df_efficiency["property"].value_counts()))

In [ ]:
battery_df_efficiency["property"].value_counts().iloc[0:25]

In [ ]:
battery_df_efficiency["unit_normalized"].value_counts()

In [ ]:
discussion.save_df(battery_df, "../data/visualisation_files/battery/df_battery.parquet")

### Hydrogen

In [29]:
result = discussion.get_metadata(
    df_quinex,
    entity_keywords=['hydrogen'],
    property_keywords=['cost', 'LCOE', 'LCOH', 'COE', 'price'],
    #top_n=100,
    exact_match=False
)

# Ausgabe
print("Total Statements:", result['total_statements'])

top_entities_names = result['top_entities'].index.tolist()
print("Number of Entities:", len(top_entities_names))

top_properties_names = result['top_properties'].index.tolist()
print("Number of Properties:", len(top_properties_names))

hydrogen_df = result["filtered_df"]

Total Statements: 3319
Number of Entities: 1387
Number of Properties: 1004


In [30]:
canonical_entities = discussion.entity_keywords["hydrogen"]

mention_to_entity = discussion.map_mentions(top_entities_names, canonical_entities, threshold=0.8)

hydrogen_df["entity_mapping"] = hydrogen_df["entity"].map(mention_to_entity)
hydrogen_df = hydrogen_df[hydrogen_df["entity_mapping"].notna()]

# Ausgabe
print("Total Statements:", len(hydrogen_df))
print("Number of Entities:", len(hydrogen_df["entity"].value_counts()))
print("Number of Properties:", len(hydrogen_df["property"].value_counts()))

Total Statements: 919
Number of Entities: 72
Number of Properties: 243


In [ ]:
hydrogen_df["property"].value_counts()

In [9]:
canonical_properties = discussion.property_keywords["cost"]

mention_to_property = discussion.map_mentions(top_properties_names, canonical_properties, threshold=0.75)

hydrogen_df["property_mapping"] = hydrogen_df["property"].map(mention_to_property)
hydrogen_df = hydrogen_df[hydrogen_df["property_mapping"].notna()]

# Ausgabe
print("Total Statements:", len(hydrogen_df))
print("Number of Entities:", len(hydrogen_df["entity"].value_counts()))
print("Number of Properties:", len(hydrogen_df["property"].value_counts()))

Total Statements: 595
Number of Entities: 51
Number of Properties: 39


C:\Users\m.gorres\AppData\Local\Temp\ipykernel_14644\742783090.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  hydrogen_df["property_mapping"] = hydrogen_df["property"].map(mention_to_property)


In [10]:
property_keywords = {
    "Levelized Cost": ["LCOH", "COE", "LCOS", "LCOE", "Levelized Cost"]
}

def map_property(prop_value):
    for category, aliases in property_keywords.items():
        if prop_value in aliases:
            return category
    return prop_value

hydrogen_df["property_mapping"] = hydrogen_df["property_mapping"].apply(map_property)
hydrogen_df["property_mapping"].value_counts()

property_mapping
Cost                392
Production Cost     122
Levelized Cost       60
Electricity Cost      9
Total Cost            5
Capital Cost          5
Operating Cost        2
Name: count, dtype: int64

In [31]:
discussion.save_df(hydrogen_df, "../data/visualisation_files/hydrogen/df_hydrogen.parquet")

💾 Saved DataFrame to '../data/visualisation_files/hydrogen/df_hydrogen.parquet' with 919 rows


### PV

In [26]:
result = discussion.get_metadata(
    df_quinex,
    entity_keywords=['PV'],
    # property_keywords=['cost'],
    # top_n=300,
    exact_match=False
)

# Ausgabe
print("Total Statements:", result['total_statements'])

top_entities_names = result['top_entities'].index.tolist()
print("Number of Entities:", len(top_entities_names))

top_properties_names = result['top_properties'].index.tolist()
print("Number of Properties:", len(top_properties_names))

pv_df = result["filtered_df"]

Total Statements: 40742
Number of Entities: 13369
Number of Properties: 15832


In [27]:
canonical_entities = discussion.entity_keywords["pv"]

mention_to_entity = discussion.map_mentions(top_entities_names, canonical_entities, threshold=0.85)

pv_df["entity_mapping"] = pv_df["entity"].map(mention_to_entity)
pv_df = pv_df[pv_df["entity_mapping"].notna()]

# Ausgabe
print("Total Statements:", len(pv_df))
print("Number of Entities:", len(pv_df["entity"].value_counts()))
print("Number of Properties:", len(pv_df["property"].value_counts()))

Total Statements: 4279
Number of Entities: 375
Number of Properties: 2115


In [28]:
discussion.save_df(pv_df, "../data/visualisation_files/pv/df_pv.parquet")

💾 Saved DataFrame to '../data/visualisation_files/pv/df_pv.parquet' with 4279 rows


### Wind Turbine

In [32]:
result = discussion.get_metadata(
    df_quinex,
    entity_keywords=['wind turbine'],
    # property_keywords=['cost'],
    # top_n=300,
    exact_match=False
)

# Ausgabe
print("Total Statements:", result['total_statements'])
top_mention_names = result['top_entities'].index.tolist()
print("Number of Entities:", len(top_mention_names))

windturbine_df = result["filtered_df"]

Total Statements: 8545
Number of Entities: 2452


In [ ]:
canonical_entities = discussion.entity_keywords["wind turbine"]
mention_to_entity = discussion.map_mentions(top_mention_names, canonical_entities, threshold=0.8)

windturbine_df["entity_mapping"] = windturbine_df["entity"].map(mention_to_entity)
windturbine_df = windturbine_df[windturbine_df["entity_mapping"].notna()]

# Ausgabe
print("Total Statements:", len(windturbine_df))
print("Number of Entities:", len(windturbine_df["entity"].value_counts()))
print("Number of Properties:", len(windturbine_df["property"].value_counts()))

In [ ]:
discussion.save_df(windturbine_df, "../data/visualisation_files/windturbine/df_windturbine.parquet")

### BESS (Battery Energy Storage System)

In [ ]:
result = discussion.get_metadata(
    df_quinex,
    entity_keywords=['BESS'],
    # property_keywords=['cost'],
    # top_n=300,
    exact_match=False
)

# Ausgabe
print("Total Statements:", result['total_statements'])
top_mention_names = result['top_entities'].index.tolist()
print("Number of Entities:", len(top_mention_names))

bess_df = result["filtered_df"]

In [ ]:
canonical_entities = discussion.entity_keywords["bess"]
mention_to_entity = discussion.map_mentions(top_mention_names, canonical_entities, threshold=0.8)

canonical_entities = discussion.entity_keywords["natural gas"]
mention_to_entity = discussion.map_mentions(top_mention_names, canonical_entities, threshold=0.8)


bess_df["entity_mapping"] = bess_df["entity"].map(mention_to_entity)
naturalgas_df = bess_df[bess_df["entity_mapping"].notna()]

# Ausgabe
print("Total Statements:", len(bess_df))
print("Number of Entities:", len(bess_df["entity"].value_counts()))
print("Number of Properties:", len(bess_df["property"].value_counts()))

### Natural Gas

In [ ]:
result = discussion.get_metadata(
    df_quinex,
    entity_keywords=['natural gas'],
    # property_keywords=['cost'],
    # top_n=300,
    exact_match=False
)

# Ausgabe
print("Total Statements:", result['total_statements'])
top_mention_names = result['top_entities'].index.tolist()
print("Number of Entities:", len(top_mention_names))

naturalgas_df = result["filtered_df"]

In [ ]:
canonical_entities = discussion.entity_keywords["natural gas"]
mention_to_entity = discussion.map_mentions(top_mention_names, canonical_entities, threshold=0.8)


naturalgas_df["entity_mapping"] = naturalgas_df["entity"].map(mention_to_entity)
naturalgas_df = naturalgas_df[naturalgas_df["entity_mapping"].notna()]

# Ausgabe
print("Total Statements:", len(naturalgas_df))
print("Number of Entities:", len(naturalgas_df["entity"].value_counts()))
print("Number of Properties:", len(naturalgas_df["property"].value_counts()))

In [ ]:
discussion.save_df(naturalgas_df, "../data/visualisation_files/naturalgas/df_naturalgas.parquet")

### Electrolyzer

In [ ]:
result = discussion.get_metadata(
    df_quinex,
    entity_keywords=['electrolyzer', 'electrolyser'],
    # property_keywords=['cost'],
    # top_n=300,
    exact_match=False
)

# Ausgabe
print("Total Statements:", result['total_statements'])
top_mention_names = result['top_entities'].index.tolist()
print("Number of Entities:", len(top_mention_names))

electrolyzer_df = result["filtered_df"]

In [ ]:
canonical_entities = discussion.entity_keywords["electrolyzer"]
mention_to_entity = discussion.map_mentions(top_mention_names, canonical_entities, threshold=0.8)


electrolyzer_df["entity_mapping"] = electrolyzer_df["entity"].map(mention_to_entity)
electrolyzer_df = electrolyzer_df[electrolyzer_df["entity_mapping"].notna()]

# Ausgabe
print("Total Statements:", len(electrolyzer_df))
print("Number of Entities:", len(electrolyzer_df["entity"].value_counts()))
print("Number of Properties:", len(electrolyzer_df["property"].value_counts()))

In [ ]:
discussion.save_df(electrolyzer_df, "../data/visualisation_files/electrolyzer/df_electrolyzer.parquet")